# Intensity-coded FMCW LiDAR walkthrough
Run from the project folder or its `notebooks/` folder. These cells call the package rather than duplicate its physics. Full campaigns run from the terminal. See README and model assumptions first.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))
from fmcw.config import load
from fmcw.experiments import analytic, demo, convergence, tradeoff
from threadpoolctl import threadpool_limits
cfg = load('config/baseline.json')
analytic(cfg)

## 1. Noiseless off-grid recovery
A single sinusoid can have precision much finer than the two-target resolution. Test a non-bin-centred range.

In [ ]:
from fmcw.simulation import simulate
from fmcw.estimation import Estimator
c = cfg.updated(range_m=50.037, noise=False, linewidth_hz=0, interferer_linewidth_hz=0)
with threadpool_limits(limits=1):
    trial = simulate(c, coded=True, seed=10)
    estimator = Estimator(c, coded=True)
    estimate, score = estimator.estimate(trial.processed)
print('Range:', estimate, 'm; error:', estimate-c.range_m, 'm')
del estimator

## 2. Independent calibration and the four comparisons
This generates one realization, not Monte Carlo statistics.

In [ ]:
with threadpool_limits(limits=1):
    rows = demo(cfg, 'results/notebook_demo')
rows

In [ ]:
from IPython.display import Image, display
display(Image(filename='results/notebook_demo/demo.png'))

## 3. Numerical and power checks

In [ ]:
with threadpool_limits(limits=1):
    print(convergence(cfg, 'results/notebook_convergence'))
    tradeoff(cfg, 'results/notebook_tradeoff')
display(Image(filename='results/notebook_tradeoff/bias_drive.png'))

## 4. Campaign analysis
Run the staged terminal commands in README. Read a completed run below; this cell does not launch a long campaign.

In [ ]:
import csv
path = Path('results/smoke_mc/summary.csv')
if path.exists():
    with path.open() as f:
        summary = list(csv.DictReader(f))
    display(summary)
    display(Image(filename=str(path.parent/'performance.png')))
else:
    print('Run the small smoke command from README first.')

## 5. Component work
Install the optional Tidy3D dependencies and run `component/solve_component.py` for both designs. Inspect mode polarization and convergence. Compare exported designs at fixed drive **volts**, then repeat the same Monte Carlo case list. The uniform RF-field approximation and unmodeled loss must accompany the component result.